In [255]:
import numpy as np
from scipy.optimize import minimize
import csv
pi = np.pi

In [256]:
def binary_search(low, high, test_func, tol=1e-7):
    """
    Generic continuous binary search engine.
    """
    while (high - low) > tol:
        mid = (low + high) / 2.0
        positive_test = test_func(mid)
        if positive_test:
            low = mid
        else:
            high = mid

    return (low + high) / 2.0

In [257]:
def binary_entropy(p):
    if p <= 0 or p >= 1:
        return 0.0
    # np.log2 works on floats; if using arrays, use np.log2
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

def C_epsilon(Epsilon): #H(ε) + H(ε')(1-ε) + ˜ε log ˜ε − ˜ε
    Epsilon_prime = Epsilon/(1-Epsilon)
    return binary_entropy(Epsilon) + binary_entropy(Epsilon_prime)*(1-Epsilon) + Epsilon*np.log(Epsilon) - Epsilon

In [258]:
def exps_bound_of_Ee(value,Delta,Zeta,Threshold=-1e-60,Epsilon=1/4):
    L_term = 3*Epsilon + 4*Zeta*(Delta-Epsilon) -2*Delta
    #linear term is δ(4ζ-2) + ε(3-4ζ)
    log_alpha = Delta*np.log(3/2)+(Epsilon-Delta)*np.log(3*Delta)
    constant_term = 2*Delta + 2*log_alpha + C_epsilon(Epsilon)

    C = np.pow(2*pi,3/2)* Delta*Epsilon*np.sqrt((1-2*Epsilon)/2)
    exp_term = -3*value/2 -np.log(C)
    result = constant_term + value*L_term + np.exp(-value)*exp_term

    return result < Threshold

In [259]:
def gamma_n(value,Zeta,Delta,Omega,Tau):
    n_value = np.exp(value)
    M1 = np.exp(value*Zeta)
    return Delta**2/4 - Delta/(4*n_value) - 16*np.power(M1,1-2*Tau)/np.power(np.log(M1),2*Omega) - 4*Delta/M1

def A_condition(value,Zeta,Delta,Epsilon,Omega,Tau,K,Eta):
    #value represents L = log n
    Gamma = gamma_n(value=value,Zeta=Zeta,Delta=Delta,Omega=Omega,Tau=Tau)
    C_k = 4*np.exp(2)*(2*K+1)**2 + 2*np.exp(2)
    M1 = np.exp(value*Zeta)
    log_M = Zeta*value

    Term1 = -Gamma/(C_k*np.power(M1,2*Tau)*np.power(log_M,2*Eta+2*Omega)) * np.power(1-1/log_M**Omega,4*K*log_M**Eta)

    Term2 = C_epsilon(Epsilon)
    #constant term

    # c=ε sqrt[2π(1 − 2ε)/2].
    c = Epsilon*np.sqrt(pi*(1-2*Epsilon)/2)
    Term3 = -value/2 - np.log(c)
    #coupled with e^-L
    Term4 = Epsilon
    #coupled with L
    return Term1*np.exp(value) +Term2+Term3*np.exp(-value)+ Term4*value < -1e-60 #+ 1/(12*Epsilon)*np.exp(-2*value)

In [260]:
def C_TX_min(value,Zeta, Eta, K,Tau): #threshold is NEGATIVE
    log_M = Zeta*value
    log_M_eta = np.pow(log_M,Eta-1)
    m_tau = np.exp(log_M*(1-2*Tau))

    terms= [
            2 - m_tau/log_M,
            K*log_M_eta,
            -K*np.log(K)*log_M_eta,
            -K*Eta*log_M_eta*np.log(log_M),
            # K*log_M_eta*np.log(m_tau),
            K*log_M_eta*(1-2*Tau)*log_M,
        ]
    # print(term1+term2-term3)
    return sum(terms)*log_M < -np.log(2)

def C_TX_condition(value,Zeta,Eta,K,Tau):
    M = np.exp(value*Zeta)
    log_M = Zeta*value
    return M-2 <= np.pow(M,2*Tau)*K*np.pow(log_M,Eta)

def Zeta_condition(value,Zeta,Tau):
    return Zeta > (2+np.log(4)/value) / (2 + Tau)

In [261]:
ZETA= 8.00973566800566416646e-01
DELTA = 4.234044105657223e-02
OMEGA= 1.022529534106635
k_param=26.618752763236166
TAU=0.505914402992784
ETA=1.691655261013411e-01


In [262]:
def theLeastL(low,high,zeta,delta,omega,k,tau,eta):
    epsilon = 1/4
    exp_term_direction = delta*(4*zeta-2) + epsilon*(3-4*zeta)>0
    # print(exp_term_direction)
    C_E = binary_search(low,high, lambda L:exps_bound_of_Ee(value=L, Epsilon=0.25,Delta=delta,Zeta=zeta,Threshold=-1e-60)==exp_term_direction)

    F_A = binary_search(low,high, lambda L:not A_condition(value=L,Zeta=zeta,Epsilon=epsilon,Delta=delta,Omega=omega,Tau=tau,K=k,Eta=eta))

    C_TX = binary_search(low,high, lambda L:C_TX_min(value=L,Zeta=zeta,Eta=eta,K=k,Tau=tau))



    Chernoff_cond = binary_search(low,high, lambda L:C_TX_condition(value=L,Zeta=zeta,Eta=eta,K=k,Tau=tau))
    zeta_cond = binary_search(low,high, lambda L:not Zeta_condition(value=L,Zeta=zeta,Tau=tau))

    floor_values= []
    ceil_values = []
    if not exp_term_direction: #if the sign of E(eps,delta) is positive, it is a ceiling value
        floor_values.append(C_E)
    else:
        ceil_values.append(C_E)

    ceil_values.append(C_TX)
    ceil_values.append(Chernoff_cond)
    floor_values.append(F_A)
    floor_values.append(zeta_cond)

    # print(f"C_E={C_E:.2f},F_A={F_A:.2f},C_TX={C_TX:.2f},Chernoff_cond={Chernoff_cond:.2f},zeta_cond={zeta_cond:.2f}")
    # print(floor_values)
    # print(ceil_values)
    if max(floor_values) - min(ceil_values) < 0:
        return max(floor_values)
    else:
        return 1e12


In [263]:
def objective(params):
    zeta, delta, omega, k, tau, eta = params
    L = theLeastL(low=100,high=300,zeta=zeta,delta=delta,omega=omega,k=k,eta=eta,tau=tau)
    return L

ZETA= 8.001e-01
DELTA = 4.34e-02
OMEGA= 1.0101
k_param=34.45
TAU=0.5061
ETA=0.1157

# Bounds for (zeta, epsilon, gamma, omega, k)
bounds = [
    (0.75, 0.9999),  # zeta: 0.8 < zeta < 1
    (0.0001,0.25), #delta: small positive
    (0.5001, 10.0),  # omega: typically > 1
    (0.001, 50000.0),  # k: k > e
    (0.45,0.99999), # tau: 0.5=< tau < 1
    (0,1),
]
# Initial guess (starting point for the optimizer)
x0 = [ZETA, DELTA, OMEGA, k_param, TAU, ETA]

# We use Nelder-Mead
res = minimize(
    objective,
    x0,
    method='Nelder-Mead',
    bounds=bounds,
    options={'maxiter': 5000, 'disp': True}
)

if res.success:
    optimized_zeta, optimized_delta, optimized_omega, optimized_k,optimized_tau,optimized_eta = res.x
    my_result = res.fun
    # Check if it actually found a valid structural constraint index
    if my_result < 1000.0:
        print(f"Minimum L found: {my_result:10f}")
        print(f"Value of n : {np.exp(res.fun):.6e}")
    else:
        print("Optimization 'converged' but only found invalid/penalized regions.")

    print(f"ZETA= {optimized_zeta:.20e}")
    # print(f"EPSILON= {optimized_epsilon:.6e}")
    print(f"DELTA = {optimized_delta:.15e}" )
    # print(f"GAMMA= {optimized_gamma:.15e}")
    print(f"OMEGA= {optimized_omega:.15f}")
    print(f"k_param={optimized_k:.15f}")
    print(f"TAU={optimized_tau:.15f}")
    print(f"ETA={optimized_eta:.15e}")
    # print(f"THRESHOLD_M={best_threshold:.5f}")


else:
    print("Optimization failed to converge.")

Optimization terminated successfully.
         Current function value: 192.884952
         Iterations: 571
         Function evaluations: 1006
Minimum L found: 192.884952
Value of n : 5.873142e+83
ZETA= 8.00945706733730222737e-01
DELTA = 4.350154394482820e-02
OMEGA= 1.011821329563779
k_param=34.301864575673264
TAU=0.506021494069048
ETA=1.175975622044602e-01


In [265]:
ZETA= 8.00945706733730222737e-01
DELTA = 4.350154394482820e-02
OMEGA= 1.011821329563779
k_param=34.301864575673264
TAU=0.506021494069048
ETA=1.175975622044602e-01
result1 = theLeastL(low=100,high=300,zeta=ZETA,delta=DELTA,omega=OMEGA,k=k_param,tau=TAU,eta=ETA)
print(result1)
# data_row = [result1,ZETA,DELTA,OMEGA,k_param,TAU,ETA]
# with open('params_opt_value.csv', 'a', newline='') as csvfile:
#     writer = csv.writer(csvfile)
#     writer.writerow(data_row)

print(exps_bound_of_Ee(156,Delta=DELTA,Zeta=ZETA))

print(f"{np.exp(result1):.4e}")

192.88495243526995
True
5.8731e+83
